In [1]:
from pathlib import Path
import json
import os
import tempfile
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import rasterio

from rasterio.transform import Affine
from tqdm.auto import tqdm


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

ROOT = Path(
    r"A:\NCA_DATA\S2_2016-2017"
)

GEE_READY_DIR = (
    ROOT / "gee_ready"
)

QA60_LOG = (
    ROOT / "qa60_reconstruction_log.csv"
)


# ---------------------------------------------------------------------
# Expected TIFF band structure
# ---------------------------------------------------------------------

OLD_BANDS = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",
    "SCL",
]

NEW_BANDS = (
    OLD_BANDS
    + ["QA60"]
)


tif_paths = sorted(
    GEE_READY_DIR.glob(
        "*_GEE.tif"
    )
)

print(
    "GEE-ready directory:",
    GEE_READY_DIR
)

print(
    "Prepared TIFFs found:",
    len(tif_paths)
)

GEE-ready directory: A:\NCA_DATA\S2_2016-2017\gee_ready
Prepared TIFFs found: 562


In [2]:
safe_zip_paths = sorted(
    ROOT.rglob("*.zip")
)

print(
    "SAFE ZIPs found:",
    len(safe_zip_paths)
)


SAFE_ZIP_INDEX = {}

for zip_path in safe_zip_paths:

    name = zip_path.name

    # Product identifier is normally the ZIP
    # filename without .zip or .SAFE.zip.
    product_id = name

    if product_id.lower().endswith(
        ".zip"
    ):
        product_id = (
            product_id[:-4]
        )

    if product_id.endswith(
        ".SAFE"
    ):
        product_id = (
            product_id[:-5]
        )

    SAFE_ZIP_INDEX[
        product_id
    ] = zip_path


print(
    "Indexed SAFE products:",
    len(SAFE_ZIP_INDEX)
)

SAFE ZIPs found: 562
Indexed SAFE products: 562


In [3]:
def find_safe_zip(
    product_id,
):
    """
    Find the retained SAFE ZIP corresponding
    to a prepared Sentinel-2 product.
    """

    if product_id in SAFE_ZIP_INDEX:
        return SAFE_ZIP_INDEX[
            product_id
        ]

    # Fallback in case ZIP naming contains
    # an additional suffix.
    matches = [
        path
        for path in safe_zip_paths
        if product_id in path.name
    ]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No SAFE ZIP found for "
            f"{product_id}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple SAFE ZIPs found "
            f"for {product_id}:\n"
            + "\n".join(
                str(p)
                for p in matches
            )
        )

    return matches[0]

In [4]:
def extract_qa60_sources(
    zip_path,
    temp_dir,
):
    """
    Extract only:

      1. QI_DATA/MSK_CLASSI_B00.jp2
      2. GRANULE/.../MTD_TL.xml

    MSK_CLASSI supplies the classification
    values.

    MTD_TL.xml supplies the authoritative
    Sentinel tile grid.
    """

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        names = z.namelist()

        classi_matches = [
            name
            for name in names
            if name.endswith(
                "MSK_CLASSI_B00.jp2"
            )
        ]

        tile_meta_matches = [
            name
            for name in names
            if (
                "/GRANULE/"
                in name.replace(
                    "\\",
                    "/"
                )
                and name.endswith(
                    "/MTD_TL.xml"
                )
            )
        ]

        if len(
            classi_matches
        ) != 1:
            raise RuntimeError(
                f"{zip_path.name}: "
                f"expected exactly one "
                f"MSK_CLASSI_B00.jp2, "
                f"found "
                f"{len(classi_matches)}"
            )

        if len(
            tile_meta_matches
        ) != 1:
            raise RuntimeError(
                f"{zip_path.name}: "
                f"expected exactly one "
                f"MTD_TL.xml, "
                f"found "
                f"{len(tile_meta_matches)}"
            )

        classi_name = (
            classi_matches[0]
        )

        tile_meta_name = (
            tile_meta_matches[0]
        )

        z.extract(
            classi_name,
            temp_dir
        )

        z.extract(
            tile_meta_name,
            temp_dir
        )

    return (
        Path(temp_dir)
        / classi_name,

        Path(temp_dir)
        / tile_meta_name,
    )

In [5]:
def local_name(
    tag,
):
    """
    Strip XML namespace from a tag.
    """
    return tag.split(
        "}"
    )[-1]


def read_tile_grid_60m(
    tile_meta_path,
):
    """
    Read the native Sentinel-2 60 m tile grid
    directly from MTD_TL.xml.

    Returns
    -------
    transform
        Rasterio Affine transform.

    nrows
        60 m tile row count.

    ncols
        60 m tile column count.

    cs_code
        CRS identifier as plain text,
        normally EPSG:32611 here.
    """

    tree = ET.parse(
        tile_meta_path
    )

    root = tree.getroot()


    # ------------------------------------------------------------
    # Horizontal CRS code
    # ------------------------------------------------------------

    cs_code = None

    for elem in root.iter():

        if (
            local_name(
                elem.tag
            )
            == "HORIZONTAL_CS_CODE"
        ):
            cs_code = (
                elem.text.strip()
            )
            break


    if cs_code is None:
        raise RuntimeError(
            "Could not find "
            "HORIZONTAL_CS_CODE "
            "in MTD_TL.xml."
        )


    # ------------------------------------------------------------
    # 60 m dimensions
    # ------------------------------------------------------------

    nrows = None
    ncols = None

    for elem in root.iter():

        if (
            local_name(
                elem.tag
            )
            == "Size"
            and elem.attrib.get(
                "resolution"
            )
            == "60"
        ):

            for child in elem:

                name = local_name(
                    child.tag
                )

                if name == "NROWS":
                    nrows = int(
                        child.text
                    )

                elif name == "NCOLS":
                    ncols = int(
                        child.text
                    )

            break


    if (
        nrows is None
        or ncols is None
    ):
        raise RuntimeError(
            "Could not find "
            "60 m raster dimensions."
        )


    # ------------------------------------------------------------
    # 60 m origin and pixel dimensions
    # ------------------------------------------------------------

    ulx = None
    uly = None
    xdim = None
    ydim = None

    for elem in root.iter():

        if (
            local_name(
                elem.tag
            )
            == "Geoposition"
            and elem.attrib.get(
                "resolution"
            )
            == "60"
        ):

            for child in elem:

                name = local_name(
                    child.tag
                )

                if name == "ULX":
                    ulx = float(
                        child.text
                    )

                elif name == "ULY":
                    uly = float(
                        child.text
                    )

                elif name == "XDIM":
                    xdim = float(
                        child.text
                    )

                elif name == "YDIM":
                    ydim = float(
                        child.text
                    )

            break


    if None in (
        ulx,
        uly,
        xdim,
        ydim,
    ):
        raise RuntimeError(
            "Incomplete 60 m "
            "geoposition metadata."
        )


    transform = Affine(
        xdim,
        0.0,
        ulx,
        0.0,
        ydim,
        uly,
    )


    return (
        transform,
        nrows,
        ncols,
        cs_code,
    )

In [6]:
def build_qa60_60m(
    classi_path,
    tile_meta_path,
):
    """
    Construct native 60 m QA60.

    MSK_CLASSI_B00.jp2:

        band 1 = opaque cloud
        band 2 = cirrus
        band 3 = snow / ice

    QA60 encoding:

        bit 10 = opaque cloud
        bit 11 = cirrus

    Therefore:

        clear          = 0
        opaque cloud   = 1024
        cirrus         = 2048
        both           = 3072

    Snow/ice is intentionally NOT encoded
    in QA60.
    """

    (
        src_transform,
        expected_rows,
        expected_cols,
        cs_code,
    ) = read_tile_grid_60m(
        tile_meta_path
    )


    with rasterio.open(
        classi_path
    ) as src:

        if src.count != 3:
            raise RuntimeError(
                "Expected "
                "MSK_CLASSI_B00 "
                "to contain 3 bands; "
                f"found {src.count}"
            )


        if (
            src.height
            != expected_rows
            or
            src.width
            != expected_cols
        ):
            raise RuntimeError(
                "MSK_CLASSI dimensions "
                "do not match the "
                "60 m MTD_TL grid:\n"
                f"mask = "
                f"{src.height} x "
                f"{src.width}\n"
                f"metadata = "
                f"{expected_rows} x "
                f"{expected_cols}"
            )


        opaque = (
            src.read(1)
            .astype(
                np.uint16
            )
        )

        cirrus = (
            src.read(2)
            .astype(
                np.uint16
            )
        )


    # ------------------------------------------------------------
    # Verify the source classification masks
    # are actually binary.
    # ------------------------------------------------------------

    opaque_values = set(
        np.unique(
            opaque
        ).tolist()
    )

    cirrus_values = set(
        np.unique(
            cirrus
        ).tolist()
    )


    if not (
        opaque_values
        <= {0, 1}
    ):
        raise RuntimeError(
            "Opaque-cloud band contains "
            "unexpected values: "
            f"{opaque_values}"
        )


    if not (
        cirrus_values
        <= {0, 1}
    ):
        raise RuntimeError(
            "Cirrus band contains "
            "unexpected values: "
            f"{cirrus_values}"
        )


    # ------------------------------------------------------------
    # QA60 bit construction
    # ------------------------------------------------------------

    qa60 = (
        (opaque << 10)
        |
        (cirrus << 11)
    ).astype(
        np.uint16
    )


    # ------------------------------------------------------------
    # Final QA60 value check
    # ------------------------------------------------------------

    qa_values = set(
        np.unique(
            qa60
        ).tolist()
    )

    allowed = {
        0,
        1024,
        2048,
        3072,
    }


    if not (
        qa_values
        <= allowed
    ):
        raise RuntimeError(
            "Constructed QA60 contains "
            "unexpected values: "
            f"{qa_values}"
        )


    return (
        qa60,
        src_transform,
        cs_code,
    )

In [7]:
def qa60_to_existing_grid(
    qa60_60m,
    src_transform,
    tif_path,
):
    """
    Map native 60 m QA60 onto the exact
    existing 10 m GeoTIFF grid.

    No CRS transformation occurs.

    Each destination pixel center is assigned
    the QA60 value of the native 60 m pixel
    containing that coordinate.

    This is equivalent to categorical
    nearest-neighbor resampling for these
    aligned north-up grids.
    """

    with rasterio.open(
        tif_path
    ) as dst:

        dst_transform = (
            dst.transform
        )

        height = (
            dst.height
        )

        width = (
            dst.width
        )


    # ------------------------------------------------------------
    # Require normal north-up grids
    # ------------------------------------------------------------

    if (
        src_transform.b != 0
        or src_transform.d != 0
        or dst_transform.b != 0
        or dst_transform.d != 0
    ):
        raise RuntimeError(
            "Rotated raster grid detected."
        )


    src_xres = (
        src_transform.a
    )

    src_yres = abs(
        src_transform.e
    )

    src_ulx = (
        src_transform.c
    )

    src_uly = (
        src_transform.f
    )


    # ------------------------------------------------------------
    # Destination pixel-center coordinates
    # ------------------------------------------------------------

    x = (
        dst_transform.c
        +
        (
            np.arange(
                width
            )
            + 0.5
        )
        * dst_transform.a
    )


    y = (
        dst_transform.f
        +
        (
            np.arange(
                height
            )
            + 0.5
        )
        * dst_transform.e
    )


    # ------------------------------------------------------------
    # Native 60 m pixel containing each
    # destination pixel center
    # ------------------------------------------------------------

    src_cols = np.floor(
        (
            x
            - src_ulx
        )
        / src_xres
    ).astype(
        np.int64
    )


    src_rows = np.floor(
        (
            src_uly
            - y
        )
        / src_yres
    ).astype(
        np.int64
    )


    # ------------------------------------------------------------
    # Bounds checks
    # ------------------------------------------------------------

    valid_cols = (
        (src_cols >= 0)
        &
        (
            src_cols
            < qa60_60m.shape[1]
        )
    )


    valid_rows = (
        (src_rows >= 0)
        &
        (
            src_rows
            < qa60_60m.shape[0]
        )
    )


    if not valid_cols.all():
        raise RuntimeError(
            "Existing TIFF extends "
            "outside native QA60 grid "
            "in X."
        )


    if not valid_rows.all():
        raise RuntimeError(
            "Existing TIFF extends "
            "outside native QA60 grid "
            "in Y."
        )


    # ------------------------------------------------------------
    # Direct categorical lookup
    # ------------------------------------------------------------

    qa60_10m = qa60_60m[
        src_rows[:, None],
        src_cols[None, :]
    ]


    return qa60_10m.astype(
        np.uint16
    )

In [8]:
def rewrite_tif_with_qa60(
    tif_path,
    qa60_10m,
):
    """
    Rewrite an existing 11-band TIFF
    as a 12-band TIFF with QA60.

    Original spectral / SCL bands are copied
    unchanged.

    QA60 becomes band 12.
    """

    tif_path = Path(
        tif_path
    )


    temp_path = (
        tif_path.parent
        /
        (
            tif_path.stem
            + "_QA60_TEMP.tif"
        )
    )


    with rasterio.open(
        tif_path
    ) as src:


        # --------------------------------------------------------
        # Restart protection
        # --------------------------------------------------------

        if (
            src.count == 12
            and
            src.descriptions[11]
            == "QA60"
        ):
            return (
                "already_complete"
            )


        if src.count != 11:
            raise RuntimeError(
                f"{tif_path.name}: "
                f"expected 11 bands, "
                f"found {src.count}"
            )


        # --------------------------------------------------------
        # Validate existing order
        # --------------------------------------------------------

        if list(
            src.descriptions
        ) != OLD_BANDS:

            raise RuntimeError(
                f"{tif_path.name}: "
                "unexpected original "
                "band structure:\n"
                f"{src.descriptions}"
            )


        if qa60_10m.shape != (
            src.height,
            src.width,
        ):
            raise RuntimeError(
                f"{tif_path.name}: "
                "QA60 dimensions do not "
                "match TIFF."
            )


        profile = (
            src.profile.copy()
        )


        profile.update(
            count=12,
            dtype="uint16",
            interleave="band",
            BIGTIFF="IF_SAFER",
        )


        with rasterio.open(
            temp_path,
            "w",
            **profile
        ) as dst:


            # ----------------------------------------------------
            # Copy original 11 bands block-by-block
            # ----------------------------------------------------

            for band_index in range(
                1,
                12
            ):

                for _, window in (
                    src.block_windows(
                        band_index
                    )
                ):

                    data = src.read(
                        band_index,
                        window=window,
                    )

                    dst.write(
                        data,
                        band_index,
                        window=window,
                    )


                dst.set_band_description(
                    band_index,
                    OLD_BANDS[
                        band_index - 1
                    ]
                )


            # ----------------------------------------------------
            # Write QA60 band
            # ----------------------------------------------------

            for _, window in (
                dst.block_windows(
                    12
                )
            ):

                row0 = int(
                    window.row_off
                )

                row1 = (
                    row0
                    + int(
                        window.height
                    )
                )

                col0 = int(
                    window.col_off
                )

                col1 = (
                    col0
                    + int(
                        window.width
                    )
                )


                dst.write(
                    qa60_10m[
                        row0:row1,
                        col0:col1,
                    ],
                    12,
                    window=window,
                )


            dst.set_band_description(
                12,
                "QA60"
            )


            # Preserve dataset-level tags.
            dst.update_tags(
                **src.tags()
            )


    # ------------------------------------------------------------
    # Validate completed temporary TIFF
    # ------------------------------------------------------------

    with rasterio.open(
        temp_path
    ) as check:


        if check.count != 12:
            raise RuntimeError(
                "Temporary TIFF has "
                "incorrect band count."
            )


        if list(
            check.descriptions
        ) != NEW_BANDS:

            raise RuntimeError(
                "Temporary TIFF has "
                "incorrect band descriptions."
            )


        qa_values = set(
            np.unique(
                check.read(12)
            ).tolist()
        )


        allowed = {
            0,
            1024,
            2048,
            3072,
        }


        if not (
            qa_values
            <= allowed
        ):
            raise RuntimeError(
                "Temporary QA60 contains "
                "unexpected values: "
                f"{qa_values}"
            )


    # ------------------------------------------------------------
    # Atomic replacement
    # ------------------------------------------------------------

    os.replace(
        temp_path,
        tif_path
    )


    return "updated"

In [10]:
def update_json_for_qa60(
    json_path,
):
    """
    Update product metadata after QA60
    has been successfully added.
    """

    json_path = Path(
        json_path
    )


    meta = json.loads(
        json_path.read_text(
            encoding="utf-8"
        )
    )


    meta["bands"] = (
        NEW_BANDS.copy()
    )


    meta[
        "qa60_reconstructed"
    ] = True


    meta[
        "qa60_source"
    ] = (
        "MSK_CLASSI_B00.jp2"
    )


    meta[
        "qa60_definition"
    ] = {
        "bit_10": (
            "opaque_cloud"
        ),
        "bit_11": (
            "cirrus"
        ),
    }


    meta[
        "qa60_native_resolution_m"
    ] = 60


    meta[
        "qa60_output_grid_m"
    ] = 10


    meta[
        "qa60_resampling"
    ] = (
        "nearest_pixel_center_lookup"
    )


    json_path.write_text(
        json.dumps(
            meta,
            indent=2
        ),
        encoding="utf-8",
    )

In [11]:
def add_qa60_to_product(
    tif_path,
):
    """
    Reconstruct and append QA60 to one
    existing Sentinel-2 prepared TIFF.

    Workflow:

      JSON product ID
          ↓
      retained SAFE ZIP
          ↓
      MSK_CLASSI_B00.jp2
          +
      MTD_TL.xml
          ↓
      native 60 m QA60
          ↓
      direct mapping to existing 10 m
      WGS84 / UTM 11N TIFF grid
          ↓
      rewrite TIFF with QA60 band 12
          ↓
      update JSON
    """

    tif_path = Path(
        tif_path
    )


    json_path = (
        tif_path.with_suffix(
            ".json"
        )
    )


    if not json_path.exists():

        raise FileNotFoundError(
            f"Missing JSON sidecar: "
            f"{json_path}"
        )


    # ------------------------------------------------------------
    # Existing TIFF check
    # ------------------------------------------------------------

    with rasterio.open(
        tif_path
    ) as src:


        if (
            src.count == 12
            and
            src.descriptions[11]
            == "QA60"
        ):

            update_json_for_qa60(
                json_path
            )

            return {
                "file": (
                    tif_path.name
                ),
                "status": (
                    "already_complete"
                ),
            }


        if src.count != 11:

            raise RuntimeError(
                f"{tif_path.name}: "
                f"expected 11 bands "
                f"before QA60, "
                f"found {src.count}"
            )


        # --------------------------------------------------------
        # Confirm prepared TIFF is still
        # WGS84 / UTM Zone 11N.
        #
        # This does not invoke a PROJ lookup.
        # --------------------------------------------------------

        crs_text = str(
            src.crs
        )


        if (
            "WGS 84 / UTM zone 11N"
            not in crs_text
            and
            "32611"
            not in crs_text
        ):

            raise RuntimeError(
                f"{tif_path.name}: "
                "unexpected output CRS:\n"
                f"{crs_text}"
            )


    # ------------------------------------------------------------
    # Read JSON metadata
    # ------------------------------------------------------------

    meta = json.loads(
        json_path.read_text(
            encoding="utf-8"
        )
    )


    product_id = meta[
        "product_id"
    ]


    # ------------------------------------------------------------
    # Find matching retained SAFE
    # ------------------------------------------------------------

    zip_path = find_safe_zip(
        product_id
    )


    # ------------------------------------------------------------
    # Build QA60
    # ------------------------------------------------------------

    with tempfile.TemporaryDirectory(
        prefix="qa60_"
    ) as temp_dir:


        (
            classi_path,
            tile_meta_path,
        ) = extract_qa60_sources(
            zip_path,
            temp_dir,
        )


        (
            qa60_60m,
            src_transform,
            cs_code,
        ) = build_qa60_60m(
            classi_path,
            tile_meta_path,
        )


        # --------------------------------------------------------
        # Confirm source tile is WGS84 / UTM 11N
        # without asking PROJ to resolve it.
        # --------------------------------------------------------

        if (
            "32611"
            not in cs_code
        ):

            raise RuntimeError(
                f"{product_id}: "
                f"unexpected Sentinel "
                f"tile CRS: "
                f"{cs_code}"
            )


        # --------------------------------------------------------
        # Native 60 m QA60 -> existing
        # exact 10 m grid
        # --------------------------------------------------------

        qa60_10m = (
            qa60_to_existing_grid(
                qa60_60m,
                src_transform,
                tif_path,
            )
        )


        # --------------------------------------------------------
        # Add QA60 as band 12
        # --------------------------------------------------------

        result = (
            rewrite_tif_with_qa60(
                tif_path,
                qa60_10m,
            )
        )


    # ------------------------------------------------------------
    # Update metadata only after successful
    # TIFF completion
    # ------------------------------------------------------------

    update_json_for_qa60(
        json_path
    )


    return {
        "file": (
            tif_path.name
        ),

        "product_id": (
            product_id
        ),

        "status": (
            result
        ),

        "safe_zip": (
            zip_path.name
        ),

        "qa60_source": (
            "MSK_CLASSI_B00.jp2"
        ),

        "grid_source": (
            "MTD_TL.xml"
        ),

        "source_crs": (
            cs_code
        ),
    }

In [12]:
test_tif = sorted(
    GEE_READY_DIR.glob(
        "*_GEE.tif"
    )
)[0]


print(
    "Testing:"
)

print(
    test_tif
)


result = add_qa60_to_product(
    test_tif
)


print(
    "\nResult:"
)

print(
    result
)

Testing:
A:\NCA_DATA\S2_2016-2017\gee_ready\S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631_GEE.tif

Result:
{'file': 'S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631_GEE.tif', 'product_id': 'S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631', 'status': 'updated', 'safe_zip': 'S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631.zip', 'qa60_source': 'MSK_CLASSI_B00.jp2', 'grid_source': 'MTD_TL.xml', 'source_crs': 'EPSG:32611'}


In [13]:
with rasterio.open(
    test_tif
) as src:

    print(
        "CRS:"
    )

    print(
        src.crs
    )


    print(
        "\nTransform:"
    )

    print(
        src.transform
    )


    print(
        "\nBand count:",
        src.count
    )


    print(
        "\nBand descriptions:"
    )

    print(
        src.descriptions
    )


    qa60 = src.read(
        12
    )


    values, counts = np.unique(
        qa60,
        return_counts=True
    )


qa_table = pd.DataFrame({
    "QA60_value": values,
    "pixel_count": counts,
})


qa_table[
    "meaning"
] = qa_table[
    "QA60_value"
].map({
    0: "clear",
    1024: "opaque cloud",
    2048: "cirrus",
    3072: "opaque cloud + cirrus",
})


display(
    qa_table
)

CRS:
PROJCS["WGS 84 / UTM zone 11N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Transform:
| 10.00, 0.00, 527240.00|
| 0.00,-10.00, 4800000.00|
| 0.00, 0.00, 1.00|

Band count: 12

Band descriptions:
('B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL', 'QA60')


,QA60_value,pixel_count,meaning
0,0,6434064,clear
1,1024,31278156,opaque cloud
2,2048,14832744,cirrus


In [14]:
test_json = (
    test_tif.with_suffix(
        ".json"
    )
)


test_meta = json.loads(
    test_json.read_text(
        encoding="utf-8"
    )
)


print(
    json.dumps(
        test_meta,
        indent=2
    )
)

{
  "product_id": "S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631",
  "mgrs_tile": "11TNH",
  "satellite": "S2A",
  "sensing_time": "2016-01-03T18:51:22+00:00",
  "system_time_start_ms": 1451847082000,
  "processing_baseline": "N0500",
  "source": "CDSE_COLLECTION1_L2A",
  "bands": [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",
    "SCL",
    "QA60"
  ],
  "aoi_buffer_meters": 500,
  "qa60_reconstructed": true,
  "qa60_source": "MSK_CLASSI_B00.jp2",
  "qa60_definition": {
    "bit_10": "opaque_cloud",
    "bit_11": "cirrus"
  },
  "qa60_native_resolution_m": 60,
  "qa60_output_grid_m": 10,
  "qa60_resampling": "nearest_pixel_center_lookup"
}


In [ ]:
#spatial sanity check with SCL

with rasterio.open(
    test_tif
) as src:

    scl = src.read(
        11
    )

    qa60 = src.read(
        12
    )


qa_cloud = (
    qa60 > 0
)


scl_cloud = np.isin(
    scl,
    [
        8,   # cloud medium probability
        9,   # cloud high probability
        10,  # thin cirrus
    ]
)


qa_pixels = int(
    qa_cloud.sum()
)


overlap_pixels = int(
    (
        qa_cloud
        & scl_cloud
    ).sum()
)


print(
    "QA60 cloudy pixels:",
    qa_pixels
)


print(
    "QA60 cloudy pixels also "
    "classified cloud/cirrus by SCL:",
    overlap_pixels
)


if qa_pixels > 0:

    print(
        "Overlap fraction:",
        overlap_pixels
        / qa_pixels
    )

QA60 cloudy pixels: 46110900
QA60 cloudy pixels also classified cloud/cirrus by SCL: 40448908
Overlap fraction: 0.8772092498736741


In [16]:
qa60_results = []


tif_paths = sorted(
    GEE_READY_DIR.glob(
        "*_GEE.tif"
    )
)


for tif_path in tqdm(
    tif_paths,
    desc="Adding QA60",
):

    try:

        result = (
            add_qa60_to_product(
                tif_path
            )
        )


    except Exception as exc:

        result = {
            "file": (
                tif_path.name
            ),

            "status": (
                "failed"
            ),

            "error": repr(
                exc
            ),
        }


    qa60_results.append(
        result
    )


qa60_df = pd.DataFrame(
    qa60_results
)


qa60_df.to_csv(
    QA60_LOG,
    index=False
)


display(
    qa60_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nLog:"
)

print(
    QA60_LOG
)

Adding QA60:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,updated,561
1,already_complete,1



Log:
A:\NCA_DATA\S2_2016-2017\qa60_reconstruction_log.csv


In [17]:
failed = qa60_df.loc[
    qa60_df[
        "status"
    ].eq(
        "failed"
    )
]


print(
    "Failures:",
    len(failed)
)


if len(
    failed
) > 0:

    display(
        failed
    )

Failures: 0


In [18]:
final_rows = []


for tif_path in tqdm(
    sorted(
        GEE_READY_DIR.glob(
            "*_GEE.tif"
        )
    ),
    desc="Final QA",
):

    try:

        with rasterio.open(
            tif_path
        ) as src:

            descriptions = list(
                src.descriptions
            )


            if src.count == 12:

                qa_values = set(
                    np.unique(
                        src.read(
                            12
                        )
                    ).tolist()
                )

            else:

                qa_values = set()


            valid_qa = (
                qa_values
                <= {
                    0,
                    1024,
                    2048,
                    3072,
                }
                and
                src.count == 12
            )


            final_rows.append({

                "file": (
                    tif_path.name
                ),

                "count": (
                    src.count
                ),

                "bands_ok": (
                    descriptions
                    == NEW_BANDS
                ),

                "qa60_values_ok": (
                    valid_qa
                ),

                "qa60_values": (
                    sorted(
                        qa_values
                    )
                ),

                "size_gib": (
                    tif_path.stat()
                    .st_size
                    / 1024**3
                ),
            })


    except Exception as exc:

        final_rows.append({

            "file": (
                tif_path.name
            ),

            "count": None,

            "bands_ok": False,

            "qa60_values_ok": False,

            "error": repr(
                exc
            ),
        })


final_qa = pd.DataFrame(
    final_rows
)


print(
    "TIFFs:",
    len(final_qa)
)


print(
    "12-band TIFFs:",
    int(
        (
            final_qa[
                "count"
            ]
            == 12
        ).sum()
    )
)


print(
    "Correct band structure:",
    int(
        final_qa[
            "bands_ok"
        ].sum()
    )
)


print(
    "Valid QA60 values:",
    int(
        final_qa[
            "qa60_values_ok"
        ].sum()
    )
)


print(
    "Total prepared volume:",
    f"{final_qa['size_gib'].sum():,.2f} GiB"
)


problem_files = final_qa.loc[
    ~(
        final_qa[
            "bands_ok"
        ]
        &
        final_qa[
            "qa60_values_ok"
        ]
    )
]


print(
    "Problem files:",
    len(problem_files)
)


if len(
    problem_files
) > 0:

    display(
        problem_files
    )

Final QA:   0%|          | 0/562 [00:00<?, ?it/s]

TIFFs: 562
12-band TIFFs: 562
Correct band structure: 562
Valid QA60 values: 562
Total prepared volume: 182.01 GiB
Problem files: 0
